# Bài học 16: Xây dựng Content Writer

**Content Writer** là agent đầu tiên trong 3 agent của sản phẩm — nó nghiên cứu chủ đề và viết bài SEO hoàn chỉnh.

Ở Mô-đun 3, bạn đã học cách xây dựng agent với tools, structured output, và chaining. Giờ bạn sẽ thấy các khái niệm đó kết hợp lại trong một **agent production** — và hiểu từng quyết định thiết kế để có thể chỉ đạo Claude Code xây dựng thứ tương tự.

Content Writer kết hợp:
- **DataForSEO search** — để nghiên cứu chủ đề (từ Bài 9)
- **Storage tools** — để lưu bài viết hoàn chỉnh xuống ổ đĩa (từ Bài 12)
- **Instructions rõ ràng** — để viết nội dung SEO có cấu trúc tốt

Một agent làm trọn vẹn công việc: nghiên cứu, viết bài, lưu trữ.

```
Chủ đề --> [Content Writer] --> web_search + write + save_article --> Bài viết đã lưu
```

> **Lưu ý**: Claude (Anthropic) hỗ trợ sử dụng `tools` và `output_schema` cùng lúc. Content Writer dùng tools nhưng xuất ra Markdown thuần, nên không cần `output_schema` ở đây.

## Tại sao một agent, không phải ba?

Bạn có thể nghĩ cần agent riêng cho nghiên cứu và agent riêng cho viết bài. Nhưng Claude Sonnet đủ năng lực để làm cả hai trong một bước:

1. Tìm kiếm trên web về chủ đề (1-2 lần tìm kiếm)
2. Xử lý kết quả tìm kiếm
3. Viết bài Markdown hoàn chỉnh
4. Lưu xuống ổ đĩa bằng `save_article()`

Cách này **đơn giản hơn** (ít file, ít bàn giao giữa agent) và **nhanh hơn** (một lần gọi agent thay vì nối chuỗi 2-3 agent).

> **Chi phí:** ~$0.10-0.30 mỗi bài viết (gọi Sonnet với search + viết bài). Mất 30-90 giây.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

## Content Writer — Code sản phẩm

Đây là định nghĩa agent thực tế từ `output/backend/agents/content_writer.py`.

Chú ý pattern:
- **Model**: Claude Sonnet (hỗ trợ tools)
- **Tools**: DataForSEO search (có điều kiện) + các hàm storage
- **Instructions**: Hướng dẫn viết bài chi tiết (cấu trúc, độ dài, SEO)

Search tool là **có điều kiện** — nếu `DATA_FOR_SEO_API_KEY` chưa được thiết lập, writer vẫn hoạt động nhưng dựa vào kiến thức sẵn có của model.

In [ ]:
# Show the actual production code
content_writer_path = os.path.abspath("../../output/backend/agents/content_writer.py")
with open(content_writer_path, "r", encoding="utf-8") as f:
    print(f.read())

## Xây dựng Content Writer từng bước

Hãy xây dựng cùng agent này từ đầu để hiểu từng phần.

### Bước 1: Thiết lập search tools (có điều kiện)

In [ ]:
from agno.agent import Agent
from agno.models.anthropic import Claude
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials
from tools.storage import save_article, list_all_articles

# Conditional search: works with or without DataForSEO
tools = [save_article, list_all_articles]
creds = get_dataforseo_credentials()
if creds:
    tools.insert(0, DataForSEOSearchTools(login=creds[0], password=creds[1]))
    print("Search tools: DataForSEO (web search enabled)")
else:
    print("Search tools: None (will write from built-in knowledge)")

print(f"Total tools: {len(tools)}")

### Bước 2: Tạo agent với instructions viết bài

In [ ]:
writer = Agent(
    name="Content Writer",
    role="Research topics and write SEO articles",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=tools,
    instructions=[
        "You research topics and write comprehensive SEO articles.",
        "RESEARCH: Do 1-2 web searches max per article.",
        "Write a well-structured Markdown article with:",
        "  - An H1 title",
        "  - 5-8 H2 section headings",
        "  - H3 subheadings where appropriate",
        "  - Bolded important keywords naturally within the text",
        "  - Bullet/numbered lists where helpful",
        "  - 1500-2500 words of engaging content",
        "After writing, call save_article with the topic, full article text, and target keywords.",
        "Never use emojis or icons.",
    ],
    markdown=True,
)

print(f"Agent: {writer.name}")
print(f"Model: Claude Sonnet")
print(f"Tools: {len(tools)}")

### Bước 3: Chạy writer

Khi bạn đưa cho writer một chủ đề, nó sẽ:
1. Tìm kiếm trên web (nếu có DataForSEO)
2. Viết bài Markdown hoàn chỉnh
3. Gọi `save_article()` để lưu vào thư mục `content/`

Bài viết được lưu tại `content/{slug}.md` và metadata tại `content/articles.json`.

In [ ]:
topic = "How to optimize on-page SEO for your website"

print(f"Writing article about: {topic}")
print("This takes 30-90 seconds...\n")

response = writer.run(f"Write an SEO article about: {topic}")
print(response.content[:2000])
if len(response.content) > 2000:
    print("\n... (truncated for display)")

In [ ]:
# Check: was the article saved?
import json
result = list_all_articles()
articles = json.loads(result)
print(f"Articles in storage: {len(articles)}")
for a in articles:
    print(f"  {a['id']}: {a['topic']} ({a['word_count']} words, {a['status']})")

## Vòng lặp phản hồi

Bạn vừa chạy writer và xem kết quả. Thực tế, kết quả đầu tiên hiếm khi hoàn hảo. Đây là cách vòng lặp hoạt động:

> **Chạy:** Bạn yêu cầu writer tạo bài viết về "on-page SEO meta tags"

> **Review:** Bài viết 1200 từ — quá ngắn. Các section H2 ổn, nhưng phần giới thiệu chung chung và không nhắc đến keyword mục tiêu đến tận đoạn 3.

> **Điều chỉnh instructions:** "Bài viết quá ngắn và phần giới thiệu yếu. Cập nhật Content Writer instructions: (1) tối thiểu 1800 từ, (2) keyword mục tiêu phải xuất hiện trong câu đầu tiên của phần giới thiệu."

> **Chạy lại:** Claude Code cập nhật `content_writer.py`, bạn chạy writer lại với cùng chủ đề. Lần này: 2100 từ, keyword ngay câu đầu.

Đây là cách bạn tinh chỉnh hành vi agent — **không phải tự sửa code**, mà bằng cách mô tả cái gì sai và "đúng" trông như thế nào. Mỗi vòng lặp mất 2-3 phút. Sau vài lần, instructions đã được tinh chỉnh xong.

**Cần chú ý gì ở output của writer:**
- **Số từ** — có nằm trong khoảng 1500-2500 theo instructions không?
- **Cấu trúc** — có H1, H2, H3 như đã chỉ định không?
- **Sử dụng keyword** — keyword mục tiêu có được in đậm và dùng tự nhiên không?
- **Lời gọi save_article()** — bài viết có thực sự được lưu vào `content/` không?

## Schema ContentOutline (phần bổ sung)

Content Writer xuất ra Markdown thuần. Nhưng structured output (đầu ra có cấu trúc) vẫn hữu ích cho mục đích khác — ví dụ khi bạn chỉ muốn tạo dàn ý trước khi viết.

Đây là schema `ContentOutline` từ Bài học 11 về structured output, minh họa cách dùng nếu bạn muốn **chỉ** lấy dàn ý có cấu trúc:

```python
class OutlineSection(BaseModel):
    heading: str = Field(description="Section heading (H2)")
    key_points: list[str] = Field(description="Bullet points to cover")

class ContentOutline(BaseModel):
    title: str = Field(description="SEO-optimized article title")
    meta_description: str = Field(description="Meta description, max 160 chars")
    target_keywords: list[str] = Field(description="Primary SEO keywords")
    sections: list[OutlineSection] = Field(description="Content sections")
```

Content Writer không dùng schema này (nó viết bài hoàn chỉnh trực tiếp). Nhưng đây là pattern hữu ích khi bạn cần cấu trúc trước nội dung.

## Tổng kết

Những gì bạn đã học:
- **Content Writer** nghiên cứu và viết bài trong một agent duy nhất
- Sử dụng **DataForSEO search** (có điều kiện) + **storage tools** (lưu, liệt kê)
- Tạo ra **bài viết Markdown** hoàn chỉnh với cấu trúc SEO
- Bài viết được **lưu cục bộ** vào thư mục `content/`
- **Đơn giản hơn nối chuỗi** — một agent đủ năng lực tốt hơn nhiều agent đơn giản
- **Các quyết định thiết kế** (conditional tools, instructions, storage integration) đều là thứ bạn có thể chỉ đạo Claude Code triển khai

**Bài tiếp theo:** Image Finder và AIO Analyzer — 2 agent hỗ trợ giúp cải thiện bài viết.

## Bài tập

Đổi biến `topic` thành chủ đề liên quan đến công việc của bạn, rồi chạy lại ô writer phía trên.

Sau khi bài viết được lưu, viết code bên dưới để:
1. Liệt kê tất cả bài viết và đếm số lượng
2. Lấy nội dung bài viết mới nhất (dùng `get_article_content` từ `tools.storage`)
3. In ra 500 ký tự đầu tiên của nội dung bài viết

In [ ]:
# Exercise: Fill in the blanks to inspect your article
from tools.storage import get_article_content
import json

# 1. List all articles
result = ___()
articles = json.loads(result)
print(f"Total articles: {___(articles)}")

# 2. Get the most recent article's content (last one in the list)
last_id = articles[-1][___]
content_json = get_article_content(last_id)
content = json.loads(content_json)

# 3. Print first 500 characters
print(f"\nArticle: {content['topic']}")
print(content['article_markdown'][:___])

<details>
<summary>Bấm để xem đáp án</summary>

```python
from tools.storage import get_article_content
import json

# 1. List all articles
result = list_all_articles()
articles = json.loads(result)
print(f"Total articles: {len(articles)}")

# 2. Get the most recent article's content
last_id = articles[-1]["id"]
content_json = get_article_content(last_id)
content = json.loads(content_json)

# 3. Print first 500 characters
print(f"\nArticle: {content['topic']}")
print(content['article_markdown'][:500])
```
</details>